In [1]:
# Imports

from pathlib import Path
from scipy.io import savemat
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import re

In [2]:
# Define participant and set paths

participant = "vp11_KL"

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError: 
    BASE_DIR = Path.cwd().parent
    
INPUT_PATH = BASE_DIR / "analysis" / "participants" / participant / "logs"
OUTPUT_PATH = BASE_DIR / "analysis" / "participants" / participant

os.chdir(INPUT_PATH)

print(BASE_DIR)
print(INPUT_PATH)
print(OUTPUT_PATH)

D:\thesis-matlab-pipeline
D:\thesis-matlab-pipeline\analysis\participants\vp11_KL\logs
D:\thesis-matlab-pipeline\analysis\participants\vp11_KL


In [3]:
# Get logs in dir

logs = [log for log in os.listdir() if log.endswith(".csv")]

print(logs)

['KL1.csv', 'KL2.csv', 'KL3.csv', 'KL4.csv', 'KL_localizer_logging.csv']


In [4]:
def build_arrays(run_type):
    match run_type:
        case "func":
            condition_order = ["coherent", "incoherent_real", "incoherent_mock", "neutral", "target"]
        case "loc":
            condition_order = ["Face", "scene", "scrambled"]

    columns = len(condition_order)

    onset_array = np.empty((columns,), dtype=np.object_)
    duration_array = np.empty((columns,), dtype=np.object_)
    names_array = np.empty((columns,), dtype=np.object_)

    return condition_order, onset_array, duration_array, names_array

In [5]:
def write_conditions_and_onsets(log, condition_order, onset_array):
    for i, condition in enumerate(condition_order):
        cond_onst = list(log[log['condition'] == condition]['s2_onset_cor'])
        cond_onset_array = np.array([[num] for num in cond_onst])
        cond_onst = [float(str(num).replace(',', '.')) for num in cond_onst]
    
        cond_onset_array = np.array([[num] for num in cond_onst], dtype=float)
        cond_onset_array = cond_onset_array + 4
        onset_array[i] = cond_onset_array

    return onset_array

In [6]:
def write_durations(duration_array):
    duration_list = [0.20, 0.20, 0.20, 0.20, 0.20]

    for i, duration in enumerate(duration_list):
        duration_array[i] = duration

    return duration_array

In [7]:
def write_names(condition_order, names_array):
    for i, name in enumerate(condition_order):
        names_array[i] = name

    return names_array

In [8]:
def to_dict(onset_array, durration_array, names_array):
    mat_dict = {
        "onsets": onset_array,
        "names": names_array,
        "durations": duration_array,
    }

    return mat_dict

In [9]:
def prep_dir(log_id: str):
    match = re.search(r"\d+", log_id)
    log_number = match.group()

    func_dir = OUTPUT_PATH / f"0{log_number}_func"
    func_dir.mkdir(parents=True, exist_ok=True)

    return func_dir

In [10]:
def save_mcf(output_dir: Path, mat_dict: dict):
    out_dir = output_dir / "mcf"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / "MCF.mat"
    savemat(out_file, mat_dict)

In [11]:
# Run conversion for all files

for log in logs:
    log_path = INPUT_PATH / log
    df = pd.read_csv(log_path)

    if re.match(r".*localizer.*\.csv", log):
        run_type = "log"
    else: 
        run_type = "func"

    condition_order, empty_onset_array, empty_duration_array, empty_name_array = build_arrays(run_type)
    
    onset_array = write_conditions_and_onsets(df, condition_order, empty_onset_array)
    duration_array = write_durations(empty_duration_array)
    names_array = write_names(condition_order, empty_name_array)
    
    mat_dict = to_dict(onset_array, duration_array, names_array)
    
    mcf_dir = prep_dir(log)
    save_mcf(mcf_dir, mat_dict)
    
    print(f"MCF for {log} saved to {mcf_dir}.")
    for array, values in mat_dict.items():
        print(f"\narray")
        print("================")
        for value in values[0:5]:
            print(value)

print(f"MCF conversion completed for {logs}.")

MCF for KL1.csv saved to D:\thesis-matlab-pipeline\analysis\participants\vp11_KL\01_func.

array
[[ 22.3169833]
 [ 40.4932857]
 [ 63.2679577]
 [ 72.5476763]
 [ 88.3083604]
 [149.101803 ]
 [174.5086005]
 [190.552125 ]
 [206.0625142]
 [248.7619492]
 [278.5166033]
 [313.2523948]
 [329.5625901]
 [337.3595184]
 [422.0590525]
 [493.014145 ]
 [524.8513942]]
[[ 13.7534854]
 [ 30.1140674]
 [198.0989529]
 [224.2717194]
 [297.0257453]
 [320.8494731]
 [432.3216724]
 [460.3604343]
 [485.8168541]
 [509.490957 ]]
[[ 55.7708332]
 [ 98.6544576]
 [124.5611919]
 [157.0322621]
 [215.4918598]
 [240.7650994]
 [288.5124272]
 [305.2890773]
 [352.5202003]
 [371.046145 ]
 [381.5419042]
 [389.5554208]
 [407.2481759]
 [440.5016717]
 [450.4976764]]
[[ 48.5069547]
 [106.7845884]
 [114.531748 ]
 [133.6078767]
 [165.012383 ]
 [258.0081514]
 [361.5498899]
 [399.0015398]
 [414.8617957]
 [500.4944371]]
[[ 80.2946899]
 [141.5214586]
 [182.2388066]
 [232.8016916]
 [345.6063691]
 [469.0236242]
 [477.8033721]
 [516.771363 ]

UnboundLocalError: local variable 'condition_order' referenced before assignment